# Gold receiving opportunity: player-game and player-team-season

**Purpose:** Support receiving and player-trend views for any targeted player, regardless of listed position. A positive offensive snap also retains a zero-target WR, TE, RB, or FB game. Seasons 2023–2026 include completed regular-season and postseason games.

| Input (Silver) | Grain | Source columns | Use |
|---|---|---:|---|
| `pbp/silver_pbp_{season}.parquet` | play | 372 | Completed-game marker, targets, receptions, air yards, team passing |
| `schedules/silver_schedules_{season}.parquet` | game | 46 | Game, week, season type, team/opponent |
| `player_stats_weekly/silver_player_stats_weekly_{season}.parquet` | player-game-team | 150 | Historical position/name and source comparison |
| `snap_counts/silver_snap_counts_{season}.parquet` | PFR player-game-team | 16 | Positive offensive-snap, zero-target observations |
| `rosters_weekly/silver_rosters_weekly_{season}.parquet` | roster player-week-team | 36 | Contextual position/name, not participation |
| `players/silver_players.parquet` | GSIS player | 39 | Exact PFR-to-GSIS bridge and name fallback |

| Output (Gold) | Grain | Columns | Use |
|---|---|---:|---|
| `receiving/player_game/gold_receiving_player_game_{season}.parquet` | `player_id + game_id + team` | 22 | Weekly player trends and game-level aDOT distribution |
| `receiving/player_team_season/gold_receiving_player_team_season_{season}.parquet` | `player_id + season + season_type + team` | 26 | Team-room comparison, observed-game and full-team-season shares |

**Transformations:** Restrict to games with a PBP `END GAME` marker; derive official receiving events and complete team passing denominators from eligible PBP; union target recipients with identified skill-position players who logged positive offensive snaps; attach contextual position/name; aggregate per player/team/season from additive game values. A traded player has a separate row for each team.

**Quality gates:** Check input and output grains, schedule/PBP joins, eligible target classification, complete denominators, source weekly-stat discrepancies, PFR bridge gaps, observed versus full-season denominators, exact aggregation and rate recomputation, no future-game leakage into player-game rows, and Parquet round trips. The PBP-derived team *passing* air-yards denominator intentionally differs from target-linked team air yards and can disagree with weekly player-stat totals. Unmapped PFR snaps are reported, never guessed by name.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'ROADMAP.md').exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError('Run from inside the NFL Dream Lab repository')
    PROJECT_ROOT = PROJECT_ROOT.parent

SILVER_ROOT = PROJECT_ROOT / 'data' / 'silver'
GOLD_ROOT = PROJECT_ROOT / 'data' / 'gold' / 'receiving'
SEASONS = (2023, 2024, 2025, 2026)
SKILL_POSITIONS = {'WR', 'TE', 'RB', 'FB'}
GAME_KEY = ['game_id', 'team']
PLAYER_GAME_KEY = ['player_id', 'game_id', 'team']
PLAYER_SEASON_KEY = ['player_id', 'season', 'season_type', 'team']

def checked_rate(numerator, denominator):
    return numerator.div(denominator.where(denominator.ne(0))).astype('float64')

def assert_unique(frame, columns, label):
    assert frame[columns].notna().all().all(), f'{label}: null grain key'
    assert not frame.duplicated(columns).any(), f'{label}: duplicate grain key'


## Load Silver inputs and identify completed games

A schedule score is not a completion gate: its update can lag PBP during the active season. `END GAME` in PBP is the publication rule. Future games remain absent until that marker arrives.


In [2]:
players = pd.read_parquet(SILVER_ROOT / 'players' / 'silver_players.parquet', columns=['gsis_id', 'pfr_id', 'display_name'])
assert_unique(players, ['gsis_id'], 'player master')
pfr_bridge = players.loc[players['pfr_id'].notna(), ['pfr_id', 'gsis_id']].copy()
assert_unique(pfr_bridge, ['pfr_id'], 'PFR bridge')
player_names = players[['gsis_id', 'display_name']].rename(columns={'gsis_id': 'player_id', 'display_name': 'master_name'})

player_game_parts = []
team_game_parts = []
quality_rows = []

for season in SEASONS:
    pbp = pd.read_parquet(SILVER_ROOT / 'pbp' / f'silver_pbp_{season}.parquet', columns=[
        'game_id', 'play_id', 'season', 'season_type', 'week', 'posteam', 'defteam',
        'play_type', 'two_point_attempt', 'pass_attempt', 'sack', 'complete_pass',
        'receiver_player_id', 'receiver_player_name', 'air_yards', 'receiving_yards', 'desc',
    ])
    schedules = pd.read_parquet(SILVER_ROOT / 'schedules' / f'silver_schedules_{season}.parquet', columns=[
        'game_id', 'season', 'week', 'game_type', 'home_team', 'away_team', 'home_score', 'away_score',
    ])
    stats = pd.read_parquet(SILVER_ROOT / 'player_stats_weekly' / f'silver_player_stats_weekly_{season}.parquet', columns=[
        'player_id', 'game_id', 'team', 'position', 'player_display_name',
        'targets', 'receptions', 'receiving_air_yards', 'passing_air_yards',
    ])
    snaps = pd.read_parquet(SILVER_ROOT / 'snap_counts' / f'silver_snap_counts_{season}.parquet', columns=[
        'game_id', 'pfr_player_id', 'team', 'position', 'offense_snaps',
    ])
    rosters = pd.read_parquet(SILVER_ROOT / 'rosters_weekly' / f'silver_rosters_weekly_{season}.parquet', columns=[
        'gsis_id', 'season', 'week', 'team', 'position', 'full_name',
    ])
    assert_unique(pbp, ['game_id', 'play_id'], f'{season} PBP')
    assert_unique(schedules, ['game_id'], f'{season} schedule')
    assert_unique(stats.loc[stats['player_id'].notna()], PLAYER_GAME_KEY, f'{season} player stats')
    assert_unique(snaps, ['game_id', 'pfr_player_id', 'team'], f'{season} snaps')
    assert_unique(rosters.loc[rosters['gsis_id'].notna()], ['gsis_id', 'season', 'week', 'team'], f'{season} rosters')

    complete_ids = pbp.loc[pbp['desc'].fillna('').str.contains('END GAME', case=False, regex=False), 'game_id'].dropna().unique()
    complete_schedule = schedules.loc[schedules['game_id'].isin(complete_ids) & schedules['game_type'].isin(['REG', 'WC', 'DIV', 'CON', 'SB'])].copy()
    assert set(complete_ids) == set(complete_schedule['game_id']), f'{season}: completed game missing schedule or unsupported type'
    pbp = pbp.loc[pbp['game_id'].isin(complete_ids)].copy()
    assert pbp['game_id'].isin(complete_schedule['game_id']).all()
    assert pbp[['season', 'season_type', 'week']].notna().all().all()
    pbp_schedule_check = pbp.merge(complete_schedule[['game_id', 'season', 'week', 'game_type']], on='game_id', validate='many_to_one', suffixes=('', '_schedule'))
    assert pbp_schedule_check['season'].eq(pbp_schedule_check['season_schedule']).all()
    assert pbp_schedule_check['week'].eq(pbp_schedule_check['week_schedule']).all()
    assert pbp_schedule_check['season_type'].eq(pbp_schedule_check['game_type'].replace({'WC': 'POST', 'DIV': 'POST', 'CON': 'POST', 'SB': 'POST'})).all()
    scored_games = complete_schedule[['home_score', 'away_score']].notna().all(axis=1).sum()

    # Every completed game contributes two team-game denominator rows, even if a team has no targets.
    home = complete_schedule[['game_id', 'season', 'week', 'home_team', 'away_team', 'game_type']].rename(columns={'home_team': 'team', 'away_team': 'opponent_team', 'game_type': 'season_type'})
    away = complete_schedule[['game_id', 'season', 'week', 'away_team', 'home_team', 'game_type']].rename(columns={'away_team': 'team', 'home_team': 'opponent_team', 'game_type': 'season_type'})
    team_games = pd.concat([home, away], ignore_index=True)
    team_games['season_type'] = team_games['season_type'].replace({'WC': 'POST', 'DIV': 'POST', 'CON': 'POST', 'SB': 'POST'})
    assert_unique(team_games, GAME_KEY, f'{season} team games')
    assert len(team_games) == 2 * len(complete_schedule)
    offense_plays = pbp.loc[pbp['posteam'].notna(), ['game_id', 'posteam']]
    assert offense_plays.merge(team_games, left_on=['game_id', 'posteam'], right_on=GAME_KEY, how='left', indicator=True, validate='many_to_one')['_merge'].eq('both').all()

    # A target is an eligible official pass directed to an identified receiver.
    eligible = pbp['posteam'].notna() & pbp['play_type'].ne('no_play') & pbp['two_point_attempt'].fillna(0).eq(0)
    eligible_plays = pbp.loc[eligible].copy()
    target_plays = eligible_plays.loc[eligible_plays['receiver_player_id'].notna()].copy()
    assert target_plays['play_type'].eq('pass').all()
    assert target_plays['pass_attempt'].eq(1).all() & target_plays['sack'].fillna(0).eq(0).all()
    assert target_plays['air_yards'].notna().all()
    target_plays['reception'] = target_plays['complete_pass'].eq(1).astype('int64')
    target_plays['receiving_yards'] = target_plays['receiving_yards'].fillna(0)
    pass_plays = eligible_plays.loc[eligible_plays['pass_attempt'].eq(1) & eligible_plays['sack'].fillna(0).eq(0) & eligible_plays['play_type'].isin(['pass', 'qb_spike'])].copy()
    # Spikes count as official pass attempts. Source pass_attempt also marks sacks and one blocked-kick play; exclude both.
    assert pass_plays['play_type'].isin(['pass', 'qb_spike']).all() & pass_plays['sack'].fillna(0).eq(0).all()
    assert target_plays[['game_id', 'play_id']].merge(pass_plays[['game_id', 'play_id']], on=['game_id', 'play_id'], how='left', indicator=True, validate='one_to_one')['_merge'].eq('both').all()
    pass_plays['passing_air_yards'] = pass_plays['air_yards'].fillna(0)
    target_plays['target'] = 1
    pass_plays['pass_attempt_count'] = 1

    player_targets = target_plays.groupby(['receiver_player_id', 'game_id', 'posteam'], as_index=False).agg(
        targets=('target', 'sum'), receptions=('reception', 'sum'), air_yards=('air_yards', 'sum'),
        receiving_yards=('receiving_yards', 'sum'), pbp_name=('receiver_player_name', 'first'),
    ).rename(columns={'receiver_player_id': 'player_id', 'posteam': 'team'})
    team_targets = target_plays.groupby(['game_id', 'posteam'], as_index=False).agg(team_targets=('target', 'sum')).rename(columns={'posteam': 'team'})
    team_passing = pass_plays.groupby(['game_id', 'posteam'], as_index=False).agg(
        team_pass_attempts=('pass_attempt_count', 'sum'), team_passing_air_yards=('passing_air_yards', 'sum'),
    ).rename(columns={'posteam': 'team'})
    team_games = team_games.merge(team_targets, on=GAME_KEY, how='left', validate='one_to_one').merge(team_passing, on=GAME_KEY, how='left', validate='one_to_one')
    for count_column in ['team_targets', 'team_pass_attempts']:
        team_games[count_column] = team_games[count_column].fillna(0).astype('int64')
    team_games['team_passing_air_yards'] = team_games['team_passing_air_yards'].fillna(0.0)
    assert (team_games['team_targets'] <= team_games['team_pass_attempts']).all()
    assert team_games['team_targets'].sum() == len(target_plays)
    assert team_games['team_pass_attempts'].sum() == len(pass_plays)

    # Unmapped PFR IDs remain diagnostic; names are not a safe identity bridge.
    snaps = snaps.loc[snaps['game_id'].isin(complete_ids) & snaps['offense_snaps'].gt(0)].copy()
    snaps = snaps.merge(pfr_bridge, left_on='pfr_player_id', right_on='pfr_id', how='left', validate='many_to_one')
    unmapped_skill_snaps = snaps.loc[snaps['gsis_id'].isna() & snaps['position'].isin(SKILL_POSITIONS)]
    snap_skill = snaps.loc[snaps['gsis_id'].notna() & snaps['position'].isin(SKILL_POSITIONS), ['gsis_id', 'game_id', 'team', 'position', 'offense_snaps']].rename(columns={'gsis_id': 'player_id', 'position': 'snap_position'})
    assert_unique(snap_skill, PLAYER_GAME_KEY, f'{season} mapped skill snaps')
    player_games = player_targets.merge(snap_skill, on=PLAYER_GAME_KEY, how='outer', validate='one_to_one')
    for count_column in ['targets', 'receptions']:
        player_games[count_column] = player_games[count_column].fillna(0).astype('int64')
    for value_column in ['air_yards', 'receiving_yards']:
        player_games[value_column] = player_games[value_column].fillna(0.0)
    player_games['observation_source'] = np.select(
        [player_games['targets'].gt(0) & player_games['offense_snaps'].notna(), player_games['targets'].gt(0)],
        ['target_and_snap', 'target'], default='offensive_snap',
    )
    player_games = player_games.merge(team_games, on=GAME_KEY, how='left', validate='many_to_one', indicator=True)
    assert player_games['_merge'].eq('both').all()
    player_games = player_games.drop(columns='_merge')
    assert_unique(player_games, PLAYER_GAME_KEY, f'{season} player games')

    stats = stats.loc[stats['game_id'].isin(complete_ids)].copy()
    team_stat_passing = stats.groupby(['game_id', 'team'], as_index=False).agg(source_team_passing_air_yards=('passing_air_yards', lambda values: values.sum(min_count=1)))
    team_air_check = team_games.merge(team_stat_passing, on=GAME_KEY, how='left', validate='one_to_one')
    air_source_present = team_air_check['source_team_passing_air_yards'].notna()
    team_air_disagreements = team_air_check.loc[air_source_present, 'team_passing_air_yards'].sub(team_air_check.loc[air_source_present, 'source_team_passing_air_yards']).abs().gt(1e-9).sum()
    stats = stats.loc[stats['player_id'].notna()].rename(columns={
        'position': 'stats_position', 'player_display_name': 'stats_name',
        'targets': 'source_targets', 'receptions': 'source_receptions',
        'receiving_air_yards': 'source_air_yards',
    })
    player_games = player_games.merge(stats, on=PLAYER_GAME_KEY, how='left', validate='one_to_one')
    rosters = rosters.loc[rosters['gsis_id'].notna(), ['gsis_id', 'season', 'week', 'team', 'position', 'full_name']].rename(columns={
        'gsis_id': 'player_id', 'position': 'roster_position', 'full_name': 'roster_name',
    })
    player_games = player_games.merge(rosters, on=['player_id', 'season', 'week', 'team'], how='left', validate='many_to_one')
    player_games = player_games.merge(player_names, on='player_id', how='left', validate='many_to_one')
    player_games['position'] = player_games['stats_position'].combine_first(player_games['roster_position']).combine_first(player_games['snap_position'])
    player_games['position_source'] = np.select(
        [player_games['stats_position'].notna(), player_games['roster_position'].notna(), player_games['snap_position'].notna()],
        ['weekly_stats', 'weekly_roster', 'offensive_snaps'], default='unknown',
    )
    name_columns = ['stats_name', 'roster_name', 'pbp_name', 'master_name']
    player_games['display_name'] = player_games[name_columns].astype('string').bfill(axis=1).iloc[:, 0]
    assert player_games['display_name'].notna().all()
    assert player_games['receptions'].le(player_games['targets']).all()
    assert player_games['targets'].le(player_games['team_targets']).all()
    assert player_games.loc[player_games['targets'].eq(0), ['receptions', 'air_yards', 'receiving_yards']].eq(0).all().all()

    player_games['target_share'] = checked_rate(player_games['targets'], player_games['team_targets'])
    player_games['air_yard_share'] = checked_rate(player_games['air_yards'], player_games['team_passing_air_yards'])
    player_games['adot'] = checked_rate(player_games['air_yards'], player_games['targets'])
    assert player_games['target_share'].dropna().between(0, 1).all()
    assert player_games.loc[player_games['targets'].gt(0), 'target_share'].notna().all()
    assert player_games.loc[player_games['targets'].eq(0), 'adot'].isna().all()

    # Source comparison is diagnostic because weekly stats can lag PBP or classify a play differently.
    source_rows = player_games.loc[player_games['source_targets'].notna()].copy()
    target_disagreements = source_rows['targets'].ne(source_rows['source_targets']).sum()
    reception_disagreements = source_rows['receptions'].ne(source_rows['source_receptions']).sum()
    air_disagreements = source_rows['air_yards'].sub(source_rows['source_air_yards']).abs().gt(1e-9).sum()
    quality_rows.append({
        'season': season, 'completed_games': len(complete_schedule), 'scored_completed_games': scored_games,
        'player_games': len(player_games), 'zero_target_snap_games': player_games['observation_source'].eq('offensive_snap').sum(),
        'unmapped_skill_snap_rows': len(unmapped_skill_snaps), 'unmapped_pfr_ids': unmapped_skill_snaps['pfr_player_id'].nunique(),
        'player_games_without_weekly_stats': player_games['source_targets'].isna().sum(),
        'target_disagreements': target_disagreements, 'reception_disagreements': reception_disagreements,
        'air_yard_disagreements': air_disagreements,
        'team_passing_air_yard_disagreements': team_air_disagreements,
        'team_games_without_passing_air_stat': (~air_source_present).sum(),
    })
    player_game_parts.append(player_games[[
        'player_id', 'season', 'week', 'season_type', 'game_id', 'team', 'opponent_team',
        'display_name', 'position', 'position_source', 'observation_source', 'offense_snaps',
        'targets', 'receptions', 'air_yards', 'receiving_yards', 'team_targets',
        'team_pass_attempts', 'team_passing_air_yards', 'target_share', 'air_yard_share', 'adot',
    ]].copy())
    team_game_parts.append(team_games)

quality_summary = pd.DataFrame(quality_rows)
display(quality_summary)


,season,completed_games,scored_completed_games,player_games,zero_target_snap_games,unmapped_skill_snap_rows,unmapped_pfr_ids,player_games_without_weekly_stats,target_disagreements,reception_disagreements,air_yard_disagreements,team_passing_air_yard_disagreements,team_games_without_passing_air_stat
0,2023,285,285,6282,1688,13,1,963,0,0,0,0,0
1,2024,285,285,6246,1792,14,3,916,0,0,0,0,0
2,2025,285,285,6220,1687,13,2,792,0,0,0,1,0
3,2026,80,79,1770,477,5,1,233,0,0,0,0,2


## Aggregate player-team-season metrics

Observed-game denominators sum only games with a player-game row. Full-team-season denominators sum every completed game for that team and season type. Rates are ratios of sums, not averages of game rates. No full-season total is attached to an earlier player-game row.


In [3]:
all_player_games = pd.concat(player_game_parts, ignore_index=True)
all_team_games = pd.concat(team_game_parts, ignore_index=True)
assert_unique(all_player_games, PLAYER_GAME_KEY, 'all player games')
assert_unique(all_team_games, GAME_KEY, 'all team games')

# The last observed position/name is contextual for the displayed team-season profile.
player_labels = (all_player_games.sort_values(['season', 'week', 'game_id'])
    .drop_duplicates(PLAYER_SEASON_KEY, keep='last')[PLAYER_SEASON_KEY + ['display_name', 'position']])
player_seasons = all_player_games.groupby(PLAYER_SEASON_KEY, as_index=False).agg(
    games_observed=('game_id', 'nunique'), targets=('targets', 'sum'),
    receptions=('receptions', 'sum'), air_yards=('air_yards', 'sum'),
    receiving_yards=('receiving_yards', 'sum'),
    team_targets_observed=('team_targets', 'sum'),
    team_pass_attempts_observed=('team_pass_attempts', 'sum'),
    team_passing_air_yards_observed=('team_passing_air_yards', 'sum'),
)
team_seasons = all_team_games.groupby(['season', 'season_type', 'team'], as_index=False).agg(
    team_games=('game_id', 'nunique'), team_targets_season=('team_targets', 'sum'),
    team_pass_attempts_season=('team_pass_attempts', 'sum'),
    team_passing_air_yards_season=('team_passing_air_yards', 'sum'),
)
player_seasons = player_seasons.merge(player_labels, on=PLAYER_SEASON_KEY, validate='one_to_one')
player_seasons = player_seasons.merge(team_seasons, on=['season', 'season_type', 'team'], how='left', validate='many_to_one', indicator=True)
assert player_seasons['_merge'].eq('both').all()
player_seasons = player_seasons.drop(columns='_merge')
player_seasons['targets_per_game'] = checked_rate(player_seasons['targets'], player_seasons['games_observed'])
player_seasons['receptions_per_game'] = checked_rate(player_seasons['receptions'], player_seasons['games_observed'])
player_seasons['adot'] = checked_rate(player_seasons['air_yards'], player_seasons['targets'])
player_seasons['target_share_observed'] = checked_rate(player_seasons['targets'], player_seasons['team_targets_observed'])
player_seasons['air_yard_share_observed'] = checked_rate(player_seasons['air_yards'], player_seasons['team_passing_air_yards_observed'])
player_seasons['target_share_full_season'] = checked_rate(player_seasons['targets'], player_seasons['team_targets_season'])
player_seasons['air_yard_share_full_season'] = checked_rate(player_seasons['air_yards'], player_seasons['team_passing_air_yards_season'])
player_seasons['team_pass_attempts_per_game'] = checked_rate(player_seasons['team_pass_attempts_season'], player_seasons['team_games'])
assert_unique(player_seasons, PLAYER_SEASON_KEY, 'player-team-season Gold')
assert player_seasons['games_observed'].le(player_seasons['team_games']).all()
assert player_seasons['team_targets_observed'].le(player_seasons['team_targets_season']).all()
assert player_seasons['team_pass_attempts_observed'].le(player_seasons['team_pass_attempts_season']).all()
assert player_seasons[['target_share_observed', 'target_share_full_season']].stack().between(0, 1).all()
assert player_seasons['target_share_full_season'].le(player_seasons['target_share_observed'] + 1e-12).all()
assert player_seasons['team_games'].gt(0).all() & player_seasons['games_observed'].gt(0).all()

SEASON_COLUMNS = [
    'player_id', 'season', 'season_type', 'team', 'display_name', 'position',
    'games_observed', 'team_games', 'targets', 'receptions', 'air_yards', 'receiving_yards',
    'team_targets_observed', 'team_pass_attempts_observed', 'team_passing_air_yards_observed',
    'team_targets_season', 'team_pass_attempts_season', 'team_passing_air_yards_season',
    'targets_per_game', 'receptions_per_game', 'adot', 'target_share_observed',
    'air_yard_share_observed', 'target_share_full_season', 'air_yard_share_full_season',
    'team_pass_attempts_per_game',
]
player_seasons = player_seasons[SEASON_COLUMNS].copy()
assert len(all_player_games.columns) == 22 and len(player_seasons.columns) == 26
display(player_seasons.loc[player_seasons['season'].eq(2026)].sort_values('targets', ascending=False).head(10))


,player_id,season,season_type,team,display_name,position,games_observed,team_games,targets,receptions,...,team_pass_attempts_season,team_passing_air_yards_season,targets_per_game,receptions_per_game,adot,target_share_observed,air_yard_share_observed,target_share_full_season,air_yard_share_full_season,team_pass_attempts_per_game
1806,00-0038543,2026,REG,SEA,Jaxon Smith-Njigba,WR,6,6,66,49,...,198,1436.0,11.0,8.166667,10.545455,0.351064,0.484680,0.351064,0.484680,33.0
1290,00-0036963,2026,REG,DET,Amon-Ra St. Brown,WR,5,5,57,37,...,191,1512.0,11.4,7.400000,8.157895,0.303191,0.307540,0.303191,0.307540,38.2
1620,00-0037744,2026,REG,ARI,Trey McBride,TE,5,5,56,42,...,200,1423.0,11.2,8.400000,5.839286,0.288660,0.229796,0.288660,0.229796,40.0
1402,00-0037239,2026,REG,NO,Chris Olave,WR,5,5,54,39,...,205,1631.0,10.8,7.800000,13.500000,0.276923,0.446965,0.276923,0.446965,41.0
1837,00-0038559,2026,REG,ARI,Michael Wilson,WR,5,5,51,29,...,200,1423.0,10.2,5.800000,10.176471,0.262887,0.364722,0.262887,0.364722,40.0
1019,00-0036358,2026,REG,DAL,CeeDee Lamb,WR,5,5,51,39,...,192,1441.0,10.2,7.800000,10.313725,0.274194,0.365024,0.274194,0.365024,38.4
1670,00-0037838,2026,REG,NYG,Isaiah Likely,TE,5,5,48,33,...,166,1562.0,9.6,6.600000,7.291667,0.294479,0.224072,0.294479,0.224072,33.2
2177,00-0039337,2026,REG,NYG,Malik Nabers,WR,5,5,46,29,...,166,1562.0,9.2,5.800000,11.804348,0.282209,0.347631,0.282209,0.347631,33.2
2129,00-0039075,2026,REG,LA,Puka Nacua,WR,3,5,45,31,...,219,1894.0,15.0,10.333333,9.666667,0.343511,0.412713,0.214286,0.229673,43.8
80,00-0031381,2026,REG,LA,Davante Adams,WR,5,5,45,27,...,219,1894.0,9.0,5.400000,15.666667,0.214286,0.372228,0.214286,0.372228,43.8


## Validate and publish

The final checks compare additive game and season components, preserve zero-target rows, and confirm a reload of every written partition. Gold outputs are local Parquet and are not committed.


In [4]:
for metric in ['targets', 'receptions', 'air_yards', 'receiving_yards']:
    assert np.isclose(player_seasons[metric].sum(), all_player_games[metric].sum())
assert np.isclose(all_player_games['targets'].sum(), all_team_games['team_targets'].sum())
assert all_player_games.groupby(['game_id', 'team'])['targets'].sum().eq(all_team_games.set_index(['game_id', 'team']).loc[all_player_games.set_index(['game_id', 'team']).index.unique(), 'team_targets']).all()
assert all_player_games.loc[all_player_games['observation_source'].eq('offensive_snap'), 'targets'].eq(0).all()

for season in SEASONS:
    game_partition = all_player_games.loc[all_player_games['season'].eq(season)].sort_values(PLAYER_GAME_KEY).reset_index(drop=True)
    season_partition = player_seasons.loc[player_seasons['season'].eq(season)].sort_values(PLAYER_SEASON_KEY).reset_index(drop=True)
    game_path = GOLD_ROOT / 'player_game' / f'gold_receiving_player_game_{season}.parquet'
    season_path = GOLD_ROOT / 'player_team_season' / f'gold_receiving_player_team_season_{season}.parquet'
    game_path.parent.mkdir(parents=True, exist_ok=True)
    season_path.parent.mkdir(parents=True, exist_ok=True)
    game_partition.to_parquet(game_path, index=False)
    season_partition.to_parquet(season_path, index=False)
    pd.testing.assert_frame_equal(pd.read_parquet(game_path), game_partition, check_dtype=True)
    pd.testing.assert_frame_equal(pd.read_parquet(season_path), season_partition, check_dtype=True)
    print(f'{season}: {len(game_partition):,} player-game rows; {len(season_partition):,} player-team-season rows')

display(quality_summary)


2023: 6,282 player-game rows; 713 player-team-season rows
2024: 6,246 player-game rows; 749 player-team-season rows
2025: 6,220 player-game rows; 764 player-team-season rows
2026: 1,770 player-game rows; 452 player-team-season rows


,season,completed_games,scored_completed_games,player_games,zero_target_snap_games,unmapped_skill_snap_rows,unmapped_pfr_ids,player_games_without_weekly_stats,target_disagreements,reception_disagreements,air_yard_disagreements,team_passing_air_yard_disagreements,team_games_without_passing_air_stat
0,2023,285,285,6282,1688,13,1,963,0,0,0,0,0
1,2024,285,285,6246,1792,14,3,916,0,0,0,0,0
2,2025,285,285,6220,1687,13,2,792,0,0,0,1,0
3,2026,80,79,1770,477,5,1,233,0,0,0,0,2


## Interpretation and caveats

Use `target_share_observed` and `air_yard_share_observed` for an active player's team-role comparison. The corresponding `*_full_season` values express his contribution to the team's entire completed season, including games he missed. `team_pass_attempts_per_game` uses the team's entire completed season; observed-game pass attempts remain available through `team_pass_attempts_observed / games_observed`. PBP-derived team passing air yards is not the same as target-linked air yards. Air-yard shares and aDOT can be negative or exceed one when air yards are signed.

Unmapped PFR snap rows cannot be assigned to a GSIS player without a defensible ID bridge. The diagnostics above quantify these omissions. Missing weekly-stat rows and PBP-versus-weekly-stat disagreements are reported, not used to overwrite canonical PBP opportunity.
